In [33]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler



# scaling the numerical data into 0~1
# Try Encoding categorical value
# mse = 0.179

In [34]:
X_full = pd.read_csv('./train.csv')
test_data = pd.read_csv('./test.csv')

y = X_full['Survived']
X_full.drop(['Survived', 'Cabin', 'Ticket', 'Name'], axis=1, inplace=True)


X_train, X_valid, y_train, y_valid = train_test_split(X_full, y, test_size=0.3, random_state=39)

In [35]:
numerical_columns = X_train.select_dtypes(include=["int64", "float64"]).columns
categorical_columns = X_train.select_dtypes(include=["category", "object"]).columns
preprocessor = ColumnTransformer([
  ('num', MinMaxScaler(), numerical_columns),
  ('cat', OneHotEncoder(handle_unknown="ignore"), categorical_columns)
])

In [36]:
model = Pipeline([
  ('process', preprocessor),
  ('classifier', RandomForestClassifier(random_state=39))
])
# model.fit(X_train, y_train)

In [ ]:
param_grid = {
    "classifier_n_estimators": [100, 300, 500],
    "classifier_max_depth": [None, 10, 20],
    "classifier_min_samples_split": [2, 5],
    "classifier_min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
  estimator=model,
  cv=3,
  n_jobs=-1,
  param_grid=param_grid
)

grid_search.fit(X_train, y_train)

ValueError: Invalid parameter 'classifier_max_depth' for estimator Pipeline(steps=[('process',
                 ColumnTransformer(transformers=[('num', MinMaxScaler(),
                                                  Index(['PassengerId', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['Sex', 'Embarked'], dtype='object'))])),
                ('classifier', RandomForestClassifier(random_state=39))]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].

In [ ]:
y_pred = grid_search.predict(X_valid)
mse = mean_squared_error(y_pred, y_valid)
mse

0.1791044776119403